# HOMER: covariance matrices and wearable summaries

The covariance example applies the same aggregation principle to subset sample covariances in Frobenius geometry. The wearable example aggregates participant summaries. Both illustrate estimation stability; neither asserts a new covariance-inference theorem or a coverage result for the wearable data.


## Choose the results to read

`paper` reads the compact manuscript summaries without rerunning experiments. Set `MODE` to `smoke` for a short numerical run or `full` for manuscript replication counts. Fresh outputs go to `runs/`; the saved figures below are displayed only in `paper` mode.


In [ ]:
MODE = "paper"  # "paper", "smoke", or "full"
STUDIES = ['covariance', 'wearable']


In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "results").is_dir() and (p / "notebooks").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook from the repository root or notebooks directory.")
if MODE not in {"paper", "smoke", "full"}:
    raise ValueError("MODE must be 'paper', 'smoke', or 'full'.")

if MODE == "paper":
    RESULTS = ROOT / "results"
    display(Markdown("**Reading the saved manuscript results.** No experiments are rerun."))
else:
    subprocess.run(
        [sys.executable, str(ROOT / "code" / "reproduce.py"), "--mode", MODE,
         "--study", *STUDIES, "--resume"],
        cwd=ROOT, check=True,
    )
    RESULTS = ROOT / "runs" / MODE / "results"
    display(Markdown(f"**Reading {MODE} results** from `runs/{MODE}/results/`."))

pd.set_option("display.max_columns", 12)
pd.set_option("display.precision", 4)


## Covariance subset estimates

Each of 32 blocks contains 64 Gaussian vectors in 12 dimensions. Every block centers its own observations before estimating covariance. Four selected blocks receive directional variance inflation, with the same observations and perturbation draws shared across methods and inflation levels.

The mean comparator averages the subset covariance matrices. It does not compute a pooled sample covariance.


In [ ]:
covariance = pd.read_csv(RESULTS / "covariance_summary.csv")
covariance["display_method"] = covariance["method"].replace({
    "Adaptive PH": "Adaptive HOMER", "GMed": "Geometric median",
})
display(covariance[["delta", "display_method", "replications", "rmse",
                    "rmse_mc_low", "rmse_mc_high", "q95_error",
                    "solver_failures", "psd_failures"]])


## Manuscript figure

The covariance heatmaps use prespecified replicate zero at the stronger perturbation, with one common color scale. The accompanying summaries use all replications.

In [ ]:
if MODE == "paper":
    display(Image(filename=str(ROOT / "figures" / "exp_covariance.png")))
else:
    display(Markdown(
        "Fresh-run summaries are shown above. To regenerate all manuscript-style "
        "figures from a complete run, use `python code/reproduce.py --mode " + MODE + " --figures`. "
        "Figure generation requires pdfLaTeX, PGF, and Poppler."
    ))


## Participant-level wearable summaries

The UCI HAR-derived inputs contain 30 participants and six activities. Within each participant and activity, preprocessing averages window-level body-acceleration energy. The target weights participants equally, and the same five blocks of six participants are used for every activity.

The analysis measures shifts under artificial perturbations. It does not evaluate classification, prediction, or confidence-interval coverage. See `../docs/DATA_SOURCES.md` and `../data/licenses/README.md` for attribution and the recorded source terms.


In [ ]:
wearable = pd.read_csv(RESULTS / "wearable_results.csv")
activities = {
    1: "Walking", 2: "Walking upstairs", 3: "Walking downstairs",
    4: "Sitting", 5: "Standing", 6: "Laying",
}
wearable["activity_name"] = wearable["activity"].map(activities)
wearable["display_method"] = wearable["method"].replace({
    "Adaptive(2)": "Adaptive HOMER", "PH(1)": "HOMER(1)",
    "PH(16)": "HOMER(16)",
})
display(wearable.groupby(["activity_name", "mechanism", "display_method"],
                         sort=False)["shift_relative_clean_mean"].max()
        .rename("largest_relative_shift_in_grid").reset_index())
print("All recorded wearable fits converged:", bool(wearable["solver_converged"].all()))


## What to take from these examples

Subset estimates need not be scalar means. Covariance matrices and participant summaries can enter the same distance-based aggregation step, with a geometry appropriate to the estimates. Robustness still depends on how many subset estimates are affected, which is why the partition and perturbation mechanism are recorded explicitly.

The covariance bootstrap intervals quantify simulation uncertainty. The wearable summaries are descriptive results from these participants. Neither should be read as an inference guarantee for adaptive HOMER.
